### Setup

In [11]:
import os
import json
import time
import pandas as pd
import torch
import numpy as np
from io import StringIO
from Bio import Entrez, SeqIO
from tqdm.auto import tqdm
from tqdm.auto import tqdm
from transformers import AutoTokenizer, EsmForMaskedLM


In [ ]:
NCBI_API_KEY = "api_key_removed_for_privacy"
Entrez.api_key = NCBI_API_KEY

Entrez.email = "rgdastidar2069@gmail.com"         
if os.environ.get("NCBI_API_KEY"):               
    Entrez.api_key = os.environ["NCBI_API_KEY"]

DELAY = 0.12 if Entrez.api_key else 0.35         
BATCH = 100                                      
CACHE = "refseq_nm_cache.json"                   

### Load the variants that need recovery

In [3]:
ref = pd.read_csv("ref_fail_windowed.tsv", sep="\t").drop(columns=["Name"], errors="ignore")
oob = pd.read_csv("pos_out_of_range.tsv", sep="\t").drop(columns=["Name"], errors="ignore")
filt = pd.read_csv("filtered_missense_grch38.tsv", sep="\t", low_memory=False,
                   usecols=["VariationID", "Name"]).drop_duplicates("VariationID")

targets = pd.concat([ref.assign(reason="wt_mismatch"),
                     oob.assign(reason="pos_out_of_range")], ignore_index=True)
targets = targets.drop_duplicates("VariationID").merge(filt, on="VariationID", how="left")
targets["nm"] = targets.Name.str.extract(r"^(NM_\d+\.\d+)")[0]

print("targets:", len(targets))
print("no NM accession in Name:", targets.nm.isna().sum())
print(targets.reason.value_counts())

targets: 4431
no NM accession in Name: 0
reason
wt_mismatch         4198
pos_out_of_range     233
Name: count, dtype: int64


### Fetch CDS translations from NCBI (cached)

In [4]:
def parse_cds(gb_text):
    """Return {versioned accession: protein translation} from a GenBank text block."""
    out = {}
    for rec in SeqIO.parse(StringIO(gb_text), "genbank"):
        for f in rec.features:
            if f.type == "CDS" and "translation" in f.qualifiers:
                out[rec.id] = f.qualifiers["translation"][0]
                break
    return out

cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
wanted = sorted(set(targets.nm.dropna()) - set(cache))
print("accessions already cached:", len(cache), "| to fetch:", len(wanted))

fetch_fail = {}
for i in tqdm(range(0, len(wanted), BATCH), desc="NCBI efetch", unit="batch"):
    ids = wanted[i:i + BATCH]
    try:
        h = Entrez.efetch(db="nuccore", id=",".join(ids), rettype="gb", retmode="text")
        text = h.read()
        h.close()
        got = parse_cds(text)
        for a in ids:
            if a in got:
                cache[a] = got[a]
            else:
                fetch_fail[a] = "no CDS translation in record"
    except Exception as e:
        for a in ids:
            fetch_fail[a] = repr(e)
    json.dump(cache, open(CACHE, "w"))   # save after each batch
    time.sleep(DELAY)

print("cached:", len(cache), "| failed:", len(fetch_fail))
for a, e in list(fetch_fail.items())[:10]:
    print(" ", a, "->", e)

accessions already cached: 0 | to fetch: 639


NCBI efetch:   0%|          | 0/7 [00:00<?, ?batch/s]

cached: 639 | failed: 0


### Check each variant against its own transcript

In [5]:
def check(nm, pos, wt):
    if pd.isna(nm):
        return "no_nm_accession", None
    seq = cache.get(nm)
    if seq is None:
        return "fetch_failed", None
    seq = seq.rstrip("*")                       # drop the stop-codon marker
    if pos < 1 or pos > len(seq):
        return "pos_out_of_range", seq
    if seq[pos - 1] != wt:
        return "wt_mismatch", seq
    return "ok", seq

res = [check(n, p, w) for n, p, w in zip(targets.nm, targets.pos, targets.wt)]
targets["status"] = [r[0] for r in res]
targets["seq"] = [r[1] for r in res]

print(pd.crosstab(targets.reason, targets.status, margins=True))
print()
print(targets.groupby("status").label.agg(["count", "mean"]).round(3))

status              ok   All
reason                      
pos_out_of_range   233   233
wt_mismatch       4198  4198
All               4431  4431

        count   mean
status              
ok       4431  0.218


### Save outputs and the recovered set

In [6]:
recovered = targets[targets.status == "ok"].copy()
print("recovered:", len(recovered), "| positive rate:", round(recovered.label.mean(), 3))
print("\nstill failing, top genes:")
print(targets[targets.status != "ok"].GeneSymbol.value_counts().head(10))

targets.drop(columns="seq").to_csv("refseq_check.tsv", sep="\t", index=False)
recovered.drop(columns="seq").to_csv("refseq_recovered.tsv", sep="\t", index=False)
print("\nsaved refseq_check.tsv and refseq_recovered.tsv")

recovered: 4431 | positive rate: 0.218

still failing, top genes:
Series([], Name: count, dtype: int64)

saved refseq_check.tsv and refseq_recovered.tsv


## Score the RefSeq set

In [ ]:
MAXRES = 1022
BS = 16
NAME = "facebook/esm2_t33_650M_UR50D"
HF_TOKEN = "hf_token_removed_for_privacy"

os.environ["HF_TOKEN"] = HF_TOKEN

d = pd.read_csv("refseq_recovered.tsv", sep="\t")
nm_seqs = json.load(open("refseq_nm_cache.json"))
d["seq"] = d.nm.map(nm_seqs).str.rstrip("*")
assert d.seq.notna().all(), "missing transcript sequence"

def window(L, pos):
    if L <= MAXRES:
        return 0, L, pos
    start = min(max(pos - 1 - MAXRES // 2, 0), L - MAXRES)
    return start, start + MAXRES, pos - start

L = d.seq.str.len().values
wins = [window(l, p) for l, p in zip(L, d.pos.values)]
d["w_start"] = [w[0] for w in wins]
d["w_end"] = [w[1] for w in wins]
d["lpos"] = [w[2] for w in wins]
d["windowed"] = (L > MAXRES).astype(int)
d["wseq"] = [s[a:b] for s, a, b in zip(d.seq, d.w_start, d.w_end)]

ok = np.array([w[lp - 1] == wt for w, lp, wt in zip(d.wseq, d.lpos, d.wt)])
assert ok.all(), f"{(~ok).sum()} wild-type mismatches; rerun Cell 4"

tok = AutoTokenizer.from_pretrained(NAME)
model = EsmForMaskedLM.from_pretrained(NAME).half().cuda().eval()
esm = model.esm
MASK = tok.mask_token_id
AA = list("ACDEFGHIKLMNPQRSTVWY")
aa_id = torch.tensor([tok.convert_tokens_to_ids(a) for a in AA], device="cuda")
aa_idx = {a: i for i, a in enumerate(AA)}

@torch.inference_mode()
def hidden(strs):
    enc = tok(strs, return_tensors="pt", padding=True).to("cuda")
    return esm(**enc).last_hidden_state

@torch.inference_mode()
def masked_logp(wseq, positions):
    base = tok(wseq, return_tensors="pt").input_ids.cuda()[0]
    P = torch.tensor(positions, device="cuda")
    batch = base.repeat(len(P), 1)
    rows = torch.arange(len(P), device="cuda")
    batch[rows, P] = MASK
    logits = model(input_ids=batch).logits[rows, P]
    return torch.log_softmax(logits.float(), dim=-1)[:, aa_id].cpu().numpy()

n = len(d)
emb = np.zeros((n, 1280), dtype=np.float16)
zs = np.full(n, np.nan, dtype=np.float32)

groups = d.groupby(["nm", "w_start"], sort=False)
for (nm, ws), g in tqdm(groups, total=groups.ngroups, desc="scoring", unit="window"):
    wseq = g.wseq.iloc[0]
    ix, lp, mt = g.index.values, g.lpos.values, g.mut.values
    wt_i = np.array([aa_idx[a] for a in g.wt.values])
    mt_i = np.array([aa_idx[a] for a in mt])

    h_wt = hidden([wseq])[0]
    for s in range(0, len(g), BS):
        sl = slice(s, s + BS)
        muts = [wseq[:p - 1] + m + wseq[p:] for p, m in zip(lp[sl], mt[sl])]
        h = hidden(muts)
        pt = torch.tensor(lp[sl], device="cuda")
        rows = torch.arange(len(muts), device="cuda")
        emb[ix[sl]] = (h[rows, pt] - h_wt[pt]).float().cpu().numpy()

    uniq = np.unique(lp)
    M = np.concatenate([masked_logp(wseq, uniq[s:s + BS].tolist())
                        for s in range(0, len(uniq), BS)], axis=0)
    u = np.searchsorted(uniq, lp)
    zs[ix] = M[u, mt_i] - M[u, wt_i]

assert not np.isnan(zs).any()
out = d.drop(columns=["seq", "wseq"])
out.to_csv("scored_refseq.tsv", sep="\t", index=False)
np.save("esm_emb_r.npy", emb)
np.save("esm_zs_r.npy", zs)
print("saved", n, "variants |", out.windowed.sum(), "windowed")

Loading weights:   0%|          | 0/571 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


scoring:   0%|          | 0/2003 [00:00<?, ?window/s]

saved 4431 variants | 2543 windowed


### Combine with the UniProt results

In [13]:
u = pd.read_csv("scored_windowed.tsv", sep="\t").assign(source="uniprot")
r = pd.read_csv("scored_refseq.tsv", sep="\t").assign(source="refseq")
both = pd.concat([u, r], ignore_index=True)
emb_all = np.concatenate([np.load("esm_emb_w.npy"), np.load("esm_emb_r.npy")])
zs_all = np.concatenate([np.load("esm_zs_w.npy"), np.load("esm_zs_r.npy")])

assert len(both) == len(emb_all) == len(zs_all)
both.to_csv("scored_combined.tsv", sep="\t", index=False)
np.save("esm_emb_all.npy", emb_all)
np.save("esm_zs_all.npy", zs_all)
print(both.source.value_counts())
print("positive rate:", round(both.label.mean(), 3))

source
uniprot    61901
refseq      4431
Name: count, dtype: int64
positive rate: 0.38


## 5-fold cross validation on combined files

In [14]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from Bio.Align import substitution_matrices

d = pd.read_csv("scored_combined.tsv", sep="\t")
emb = np.load("esm_emb_all.npy").astype(np.float32)
zs = np.load("esm_zs_all.npy")
y = d.label.values
groups = d.GeneSymbol.values

B = substitution_matrices.load("BLOSUM62")
blosum = np.array([B[w][m] for w, m in zip(d.wt, d.mut)])

subsets = {"all": np.ones(len(d), bool),
           "full-length": d.windowed.values == 0,
           "windowed": d.windowed.values == 1}

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
records, oof = [], np.full(len(d), np.nan)

for k, (tr, te) in enumerate(sgkf.split(emb, y, groups)):
    assert set(groups[tr]).isdisjoint(groups[te])
    sc = StandardScaler().fit(emb[tr])
    clf = LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced")
    clf.fit(sc.transform(emb[tr]), y[tr])
    oof[te] = clf.decision_function(sc.transform(emb[te]))

    scores = {"BLOSUM62": -blosum, "ESM-2 zero-shot": -zs,
              "ESM-2 embedding + logreg": oof}
    for sname, mask in subsets.items():
        t = te[mask[te]]
        if len(np.unique(y[t])) < 2:
            continue
        for mname, s in scores.items():
            records.append(dict(fold=k, subset=sname, method=mname, n=len(t),
                                pos_rate=y[t].mean(),
                                AUROC=roc_auc_score(y[t], s[t]),
                                PRAUC=average_precision_score(y[t], s[t])))

res = pd.DataFrame(records)
summary = (res.groupby(["subset", "method"])
              .agg(n=("n", "mean"), pos_rate=("pos_rate", "mean"),
                   AUROC_mean=("AUROC", "mean"), AUROC_sd=("AUROC", "std"),
                   PRAUC_mean=("PRAUC", "mean"), PRAUC_sd=("PRAUC", "std"))
              .round(3))
print(summary.to_string())
res.to_csv("cv_results_combined.tsv", sep="\t", index=False)

/tmp/ipykernel_2053233/4053578845.py:8: DtypeWarning: Columns (12,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  d = pd.read_csv("scored_combined.tsv", sep="\t")


                                            n  pos_rate  AUROC_mean  AUROC_sd  PRAUC_mean  PRAUC_sd
subset      method                                                                                 
all         BLOSUM62                  13266.4     0.380       0.663     0.005       0.504     0.038
            ESM-2 embedding + logreg  13266.4     0.380       0.897     0.005       0.863     0.014
            ESM-2 zero-shot           13266.4     0.380       0.901     0.007       0.875     0.013
full-length BLOSUM62                   7186.2     0.471       0.654     0.007       0.589     0.035
            ESM-2 embedding + logreg   7186.2     0.471       0.883     0.011       0.875     0.020
            ESM-2 zero-shot            7186.2     0.471       0.890     0.011       0.888     0.013
windowed    BLOSUM62                   6080.2     0.270       0.682     0.020       0.395     0.086
            ESM-2 embedding + logreg   6080.2     0.270       0.906     0.008       0.835     0.047


## Bootstrap confidence intervals

In [ ]:
n  pos_rate  AUROC_mean  AUROC_sd  PRAUC_mean  PRAUC_sd
subset      method                                                                                 
all         BLOSUM62                  13266.4     0.380       0.663     0.005       0.504     0.038
            ESM-2 embedding + logreg  13266.4     0.380       0.897     0.005       0.863     0.014
            ESM-2 zero-shot           13266.4     0.380       0.901     0.007       0.875     0.013
full-length BLOSUM62                   7186.2     0.471       0.654     0.007       0.589     0.035
            ESM-2 embedding + logreg   7186.2     0.471       0.883     0.011       0.875     0.020
            ESM-2 zero-shot            7186.2     0.471       0.890     0.011       0.888     0.013
windowed    BLOSUM62                   6080.2     0.270       0.682     0.020       0.395     0.086
            ESM-2 embedding + logreg   6080.2     0.270       0.906     0.008       0.835     0.047
            ESM-2 zero-shot            6080.2     0.270       0.910     0.008       0.851     0.037